In [17]:
import os
os.makedirs("../models", exist_ok=True)

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
data = pd.read_csv("../data/final_dataset.csv")
data.head()

,year,rank_diff,age_diff,ace_diff,df_diff,bp_saved_diff,p1_rank,p2_rank,p1_age,p2_age,...,round_ER,round_F,round_QF,round_R128,round_R16,round_R32,round_R64,round_RR,round_SF,best_of_5
0,2000,52.0,9.4,-11.0,4.0,1.0,63.0,11.0,31.1,21.7,...,False,False,False,False,False,True,False,False,False,False
1,2000,162.0,0.2,0.0,-7.0,2.0,211.0,49.0,24.5,24.3,...,False,False,False,False,False,True,False,False,False,False
2,2000,-11.0,-5.2,0.0,-6.0,-6.0,48.0,59.0,21.3,26.5,...,False,False,False,False,False,True,False,False,False,False
3,2000,-16.0,1.5,-6.0,-1.0,-6.0,45.0,61.0,19.9,18.4,...,False,False,False,False,False,True,False,False,False,False
4,2000,-133.0,-3.6,7.0,6.0,-1.0,34.0,167.0,23.7,27.3,...,False,False,False,False,False,True,False,False,False,False


In [3]:
train = data[data["year"] <= 2015]
val   = data[(data["year"] >= 2016) & (data["year"] <= 2017)]
test  = data[data["year"] >= 2018]

X_train = train.drop(columns=["y","year"])
y_train = train["y"]

X_val = val.drop(columns=["y","year"])
y_val = val["y"]

X_test = test.drop(columns=["y","year"])
y_test = test["y"]

In [4]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,10
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [5]:
def evaluate(model, X, y, name=""):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:,1]

    acc = accuracy_score(y, y_pred)
    f1 = f1_score(y, y_pred)
    auc = roc_auc_score(y, y_proba)

    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.3f}")
    print(f"F1-score: {f1:.3f}")
    print(f"ROC-AUC: {auc:.3f}")


evaluate(rf, X_train, y_train, "RF TRAIN")
evaluate(rf, X_val, y_val, "RF VAL")
evaluate(rf, X_test, y_test, "RF TEST")


=== RF TRAIN ===
Accuracy: 0.878
F1-score: 0.878
ROC-AUC: 0.958

=== RF VAL ===
Accuracy: 0.737
F1-score: 0.737
ROC-AUC: 0.808

=== RF TEST ===
Accuracy: 0.724
F1-score: 0.724
ROC-AUC: 0.793


In [13]:
rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)

In [7]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)

xgb.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


In [8]:
evaluate(xgb, X_train, y_train, "XGB TRAIN")
evaluate(xgb, X_val, y_val, "XGB VAL")
evaluate(xgb, X_test, y_test, "XGB TEST")


=== XGB TRAIN ===
Accuracy: 0.772
F1-score: 0.773
ROC-AUC: 0.854

=== XGB VAL ===
Accuracy: 0.735
F1-score: 0.737
ROC-AUC: 0.807

=== XGB TEST ===
Accuracy: 0.724
F1-score: 0.724
ROC-AUC: 0.795


In [14]:
xgb_pred = xgb.predict(X_test)
xgb_proba = xgb.predict_proba(X_test)[:, 1]

xgb_acc = accuracy_score(y_test, xgb_pred)
xgb_f1 = f1_score(y_test, xgb_pred)
xgb_auc = roc_auc_score(y_test, xgb_proba)

In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

input_dim = X_train.shape[1]

nn = Sequential([
    Dense(64, activation='relu', input_shape=(input_dim,)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

nn.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

C:\Users\George\anaconda3\envs\tennis_env\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
history = nn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=1024,
    verbose=1
)

Epoch 1/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5663 - loss: 3.0711 - val_accuracy: 0.6176 - val_loss: 0.8938
Epoch 2/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5879 - loss: 1.1229 - val_accuracy: 0.5799 - val_loss: 0.6775
Epoch 3/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5948 - loss: 0.7972 - val_accuracy: 0.5574 - val_loss: 0.6648
Epoch 4/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5978 - loss: 0.7094 - val_accuracy: 0.5704 - val_loss: 0.6539
Epoch 5/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5990 - loss: 0.6855 - val_accuracy: 0.5898 - val_loss: 0.6489
Epoch 6/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6237 - loss: 0.6657 - val_accuracy: 0.6315 - val_loss: 0.6379
Epoch 7/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6412 - loss: 0.6462 - val_accuracy: 0.6554 - val_loss: 0.6196
Epoch 8/15
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6519 - loss: 0.6418 - val_accuracy: 0.6751 - val_loss

In [11]:
# eval
y_proba = nn.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("NN Test Accuracy:", accuracy_score(y_test, y_pred))
print("NN Test F1:", f1_score(y_test, y_pred))
print("NN Test ROC-AUC:", roc_auc_score(y_test, y_proba))

471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 719us/step
NN Test Accuracy: 0.7101468536115356
NN Test F1: 0.7100119664938173
NN Test ROC-AUC: 0.7761419823846282


In [15]:
nn_proba = nn.predict(X_test).ravel()
nn_pred = (nn_proba >= 0.5).astype(int)

nn_acc = accuracy_score(y_test, nn_pred)
nn_f1 = f1_score(y_test, nn_pred)
nn_auc = roc_auc_score(y_test, nn_proba)

471/471 ━━━━━━━━━━━━━━━━━━━━ 0s 453us/step


In [16]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost", "Neural Network"],
    "Accuracy": [0.717, rf_acc, xgb_acc, nn_acc],
    "F1":       [0.715, rf_f1, xgb_f1, nn_f1],
    "ROC-AUC":  [0.780, rf_auc, xgb_auc, nn_auc]
})

results

,Model,Accuracy,F1,ROC-AUC
0,Logistic Regression,0.717000,0.715000,0.780000
1,Random Forest,0.724367,0.723982,0.793062
2,XGBoost,0.724434,0.723846,0.795331
3,Neural Network,0.710147,0.710012,0.776142


In [19]:
import joblib

joblib.dump(rf, "../models/rf.pkl")
joblib.dump(xgb, "../models/xgb.pkl")

print("Models saved!")

Models saved!
